In [2]:
import tensorflow as tf
import numpy as np
import pandas as pd

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

TensorFlow version: 2.21.0
NumPy version: 2.4.4
Pandas version: 3.0.2


NumPy: 다차원 배열 기반의 수치 연산 라이브러리. 머신러닝 데이터 표현·연산의 기반임.

Pandas: 표 형태(행/열) 데이터를 다루는 라이브러리. CSV 같은 정형 데이터를 읽고 가공하는 데 씀.

TensorFlow: 구글이 만든 딥러닝 프레임워크. 텐서 연산, 자동 미분, GPU 가속을 제공함.


In [ ]:
path_to_train = tf.keras.utils.get_file(
    "train.txt", "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
)
path_to_test = tf.keras.utils.get_file(
    "test.txt", "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt"
)

14628807/14628807 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
4893335/4893335 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Keras는 신경망을 쉽게 정의할 수 있게 해주는 고수준 API임. 레이어를 블록처럼 쌓아서 모델을 만듦.

지금은 TensorFlow 안에 tf.keras로 통합되어 있음. TensorFlow가 연산 엔진, Keras가 그 위의 인터페이스라고 보면 됨. 그래서 tf.keras.utils.get_file()처럼 바로 호출 가능함.


In [4]:
train_text = open(path_to_train, "rb").read().decode(encoding="utf-8")
test_text = open(path_to_test, "rb").read().decode(encoding="utf-8")

print("Length of train text: {}".format(len(train_text)))
print("Length of test text: {}".format(len(test_text)))
print(train_text[:300])

Length of train text: 6937271
Length of test text: 2318260
id	document	label
9976970	아 더빙.. 진짜 짜증나네요 목소리	0
3819312	흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나	1
10265843	너무재밓었다그래서보는것을추천한다	0
9045019	교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정	0
6483659	사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 던스트가 너무나도 이뻐보였다	1
5403919	막 걸음마 뗀 3세부터 초등학교 1학년생인 8살용영화.ㅋㅋㅋ...별반개도 아까움.	0
7797314	원작의


훈련 데이터: 모델이 패턴을 학습할 때 쓰는 데이터.
테스트 데이터: 학습이 끝난 모델이 새로운 데이터에서도 잘 동작하는지 평가하는 데이터. 학습에는 절대 사용하지 않음.

이렇게 분리해야 모델의 일반화 성능을 정직하게 측정할 수 있음.

과적합(overfitting): 모델이 훈련 데이터에 지나치게 맞춰져, 훈련 정확도는 높지만 테스트에서는 성능이 떨어지는 현상. 훈련/테스트 점수 차이가 크면 과적합을 의심해야 함.


In [5]:
# 정답 라벨 만들기
train_Y = np.array(
    [
        [int(row.split("\t")[2])]
        for row in train_text.split("\n")[1:]
        if row.count("\t") > 0
    ]
)

test_Y = np.array(
    [
        [int(row.split("\t")[2])]
        for row in test_text.split("\n")[1:]
        if row.count("\t") > 0
    ]
)

print("Train Y Shape: {}".format(train_Y.shape))
print("Test Y Shape: {}".format(test_Y.shape))
print("Train Y Sample: {}".format(train_Y[:5]))

Train Y Shape: (150000, 1)
Test Y Shape: (50000, 1)
Train Y Sample: [[0]
 [1]
 [0]
 [0]
 [1]]


In [6]:
import re


def clean_str(string):
    string = re.sub(r"[^가-힣A-Za-z0-9(),!?\'\`]", " ", string)
    string = re.sub(r"\'s", " 's", string)
    string = re.sub(r"\'ve", " 've", string)
    string = re.sub(r"n\'t", " n't", string)
    string = re.sub(r"\'re", " 're", string)
    string = re.sub(r"\'d", " 'd", string)
    string = re.sub(r"\'ll", " 'll", string)
    string = re.sub(r",", " , ", string)
    string = re.sub(r"!", " ! ", string)
    string = re.sub(r"\(", " \( ", string)
    string = re.sub(r"\)", " \) ", string)
    string = re.sub(r"\?", " \? ", string)
    string = re.sub(r"\s{2,}", " ", string)
    string = re.sub(r"\'{2,}", "'", string)
    string = re.sub(r"\'", "", string)

    return string.lower()


train_text_X = [
    row.split("\t")[1] for row in train_text.split("\n")[1:] if row.count("\t") > 0
]
train_text_X = [clean_str(sentence) for sentence in train_text_X]
# 문장을 띄어쓰기 단위로 단어 분리
sentences = [sentence.split(" ") for sentence in train_text_X]
for i in range(5):
    print(sentences[i])

['아', '더빙', '진짜', '짜증나네요', '목소리']
['흠', '포스터보고', '초딩영화줄', '오버연기조차', '가볍지', '않구나']
['너무재밓었다그래서보는것을추천한다']
['교도소', '이야기구먼', '솔직히', '재미는', '없다', '평점', '조정']
['사이몬페그의', '익살스런', '연기가', '돋보였던', '영화', '!', '스파이더맨에서', '늙어보이기만', '했던', '커스틴', '던스트가', '너무나도', '이뻐보였다']


지도학습: 입력 X와 정답 Y가 짝지어진 데이터로 학습하는 방식. 모델이 X로 예측값을 내고, 정답 Y와의 오차를 줄이는 방향으로 가중치를 업데이트함. 이번 실습의 train_X(문장)와 train_Y(0/1 라벨)가 정확히 이 구조임.

비지도학습: 정답 Y 없이 데이터 자체의 구조나 패턴을 찾는 방식. 군집화·차원 축소가 대표적임.

차이: 지도학습은 정답을 보고 배우고, 비지도학습은 정답 없이 숨은 구조를 찾음. NSMC는 라벨이 있으니 지도학습임.


In [8]:
VOCAB_SIZE = 2000
MAX_LEN = 25

vectorize_layer = tf.keras.layers.TextVectorization(
    standardize="lower_and_strip_punctuation",
    split="whitespace",
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_LEN,
)
vectorize_layer.adapt(train_text_X)

train_X = vectorize_layer(train_text_X)

print(train_X[:5])

tf.Tensor(
[[  23  902    5    1 1097    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0]
 [ 586    1    1    1    1    1    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0]
 [   1    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0]
 [   1    1   68  345   28   33    1    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0]
 [   1    1  102    1    2    1    1  844    1    1  570    1    0    0
     0    0    0    0    0    0    0    0    0    0    0]], shape=(5, 25), dtype=int64)


전처리: 원본 데이터를 모델이 학습하기 좋은 형태로 다듬는 과정. 결측치 처리, 정규화, 토큰화, 인코딩 등이 포함됨.

TextVectorization: 텍스트 전처리를 한 번에 처리하는 Keras 레이어. 소문자 변환, 구두점 제거, 토큰 분리, 단어 사전 학습, 정수 인코딩, 길이 통일까지 한 번에 수행함.

벡터화가 필요한 이유: 신경망은 숫자만 계산할 수 있음. 문자열을 그대로 넣을 수 없으니 단어를 정수 ID로 매핑해야 함. 예: "이 영화 정말 재미있다" → [2, 5, 9, 11].

패딩: 문장마다 길이가 다르면 텐서로 묶을 수 없음. 짧은 문장 뒤에 0을 채우고, 긴 문장은 잘라 길이를 통일함. 이번 코드에서는 MAX_LEN=25로 맞춤.


In [ ]:
import matplotlib.pyplot as plt

sentence_lengths = [len(sentence) for sentence in sentences]
sentence_lengths.sort()
plt.plot(sentence_lengths)
plt.show()

print(sum([int(l <= MAX_LEN) for l in sentence_lengths]))

Matplotlib: 파이썬의 대표 시각화 라이브러리. 선 그래프, 막대 그래프, 산점도, 히스토그램 등을 그릴 수 있음. 보통 import matplotlib.pyplot as plt로 불러와 plt.plot(), plt.show() 형태로 사용함. 머신러닝에서는 데이터 분포 확인, 학습 곡선 시각화, 예측 결과 비교 등에 자주 씀.
